In [ ]:
!jq ".[$(jq 'keys[]' ../data/dataset.json | shuf -n 1)]" ../data/dataset.json

In [ ]:

!jq ".[$(jq 'keys[]' ../data/queries.json | shuf -n 1)]" ../data/queries.json

In [ ]:
import json 
from pathlib import Path 
from tinydb import TinyDB, Query
import re
import numpy as np


db = TinyDB('db.json')
ROOT = Path.cwd().parent 
dataset_path = ROOT / "data" / "dataset.json"
query_path = ROOT / "data" / "nomic_embs" / "queries_emb_768.npy"
query_index_path = ROOT / "data" / "nomic_embs" / "queries_index.json"

DOC_PREFIX = "search_document: "

In [ ]:
def clean(text: str) -> str:
    """Remove {{placeholder}} tokens, collapse whitespace."""
    return re.sub(r"\s+", " ", re.sub(r"\{\{[^}]+\}\}", "", text)).strip()


def is_weak_title(el: dict) -> bool:
    """True when title has ≤4 words or shares no token with its own tags/category."""
    t = el["title"].lower()
    signals = el.get("tags", []) + [el.get("category", ""), el.get("subcategory", "")]
    has_signal = any(s.lower().replace("-", " ") in t for s in signals if s)
    return len(el["title"].split()) <= 4 or not has_signal


def extract_structured(el: dict) -> str:
    """
    Richest form: explicit field labels + full content.
    Named fields (Title:, Category:, Tags:) guide nomic attention to distribute
    semantic load evenly across ALL Matryoshka bands.
    Primary fix for Type-A failures (branding, coaching, communication, audit):
    engine is completely blind — needs every contextual signal available.
    """
    title  = el["title"].strip()
    tags   = ", ".join(el["tags"])
    body   = clean(el["content"])
    cat    = el["category"]
    subcat = el.get("subcategory", "")
    diff   = el.get("difficulty", "general")
    ph     = el.get("placeholders", [])
    ph_str = f"Variables: {', '.join(ph)}.\n" if ph else ""

    return (
        DOC_PREFIX
        + f"Title: {title}\n"
          f"Category: {cat} > {subcat}\n"
          f"Difficulty: {diff}\n"
          f"Tags: {tags}\n"
        + ph_str
        + body
    ).strip()


In [ ]:
with open(dataset_path, "r") as f:
    dataset = json.load(f)

In [ ]:

prompts = Query()

In [ ]:


for el in dataset:
    prompt = extract_structured(el)
    record = {
        'id'         : el['id'],
        'doc_string' : prompt,
    }
    # ── store raw metadata fields for salience weighting ──
    for field in [
        'upvotes', 'likes', 'author_reputation', 'views',
        'version', 'fork_count', 'uses', 'created_at',
        'category', 'subcategory', 'difficulty', 'language', 'target_model'
    ]:
        if field in el:
            record[field] = el[field]
    db.insert(record)


In [ ]:
db.search(prompts.id == "pk_03138")

### I ran this part on google colab due to gpu issues, I used the vs code extention for google colab.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install tinydb

In [ ]:
!ls /content/drive/MyDrive/prompt_kaban

In [ ]:
%cd /content/drive/MyDrive/prompt_kaban

In [ ]:
!uv run python nomic_corpus_embs.py \
  --dims 768 \
  --source-db db.json \
  --output-dir results

In [ ]:
!uv run python get_query_emb_nomic.py\
  --dims 768 \
  --output-dir results

## Load embeddings and build ArrowSpace

### Data Pipeline

TinyDB (db.json)

└── id + doc_string (20k records, sorted by id)

│
▼
nomic_corpus_embs.py ← encodes doc_strings, saves:

├── results/embeddings_nomic_structured_768d_raw.npy (N, 768) float32
├── results/embeddings_nomic_structured_768d_ids.npy (N,) str
└── results/embeddings_registry.json id → row_idx lookup
│
▼
ArrowSpaceBuilder.build(rows)
├── aspace ← spectral index with per-row λ (Rayleigh quotient)
└── gl ← GraphLaplacian (reused at query time)


Row alignment is guaranteed: `ids[i]` always corresponds to `embs[i]` and
`aspace.lambdas[i]`. The sort by id at save time makes this stable across runs.

> **⚠️ RAM budget (float64 in-memory)**
>
> | N items | 256d  | 512d  | 768d   |
> |---------|-------|-------|--------|
> | 20 k    | 40 MB | 80 MB | 120 MB |
> | 100 k   | 200 MB| 400 MB| 600 MB |
> | 500 k   | 1 GB  | 2 GB  | 3 GB   |
> | **1 M** | **2 GB** | **4 GB** | **6 GB** |
>
> ArrowSpace keeps a second copy of the matrix internally → **double the above**.
> For 1M prompts use **256d on a 16 GB machine**, or **768d on a 32 GB machine**.
> The Laplacian (sparse, k=6 neighbours) adds ~50 MB regardless of N.

In [ ]:
import numpy as np
from pathlib import Path

EMBS_DIR = ROOT / "data" / "nomic_embs" 
DIM      = 768  

emb_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_raw.npy"
ids_path = EMBS_DIR / f"embeddings_nomic_structured_{DIM}d_ids.npy"

# ArrowSpace requires float64
embs = np.load(emb_path).astype(np.float64)
ids  = list(np.load(ids_path))

# alignment guard — must never fail
assert embs.shape[0] == len(ids), \
    f"Shape mismatch: embs {embs.shape[0]} rows vs {len(ids)} ids"

print(f"Embeddings : {embs.shape}  dtype={embs.dtype}")
print(f"IDs        : {len(ids)} entries")
print(f"RAM in use : {embs.nbytes / 1e9:.2f} GB")
print(f"Sample     : ids[0]={ids[0]!r}  embs[0,:4]={embs[0, :4]}")

## Metadata Salience — Spectral-Aware Engagement Weighting

Rather than post-hoc blending scores, we **reshape the ArrowSpace graph topology**
before building the Laplacian. High-engagement prompts are given a larger embedding
radius, so they acquire more kNN edges and accumulate higher Rayleigh-quotient λ values.
At query time, taumode scoring propagates that signal automatically — no reranker needed.

### Salience formula

```
salience = α·norm(upvotes) + β·norm(likes) + γ·norm(author_reputation) + δ·log1p(views)
scale     = floor + (1 − floor) · salience     ∈ [floor, 1+floor]
rows_meta = embs_base × scale[:, None]
```

The `floor=0.85` parameter ensures zero-engagement prompts still retain 85 % of their
semantic signal — relevance is *diminished*, never zeroed out.


In [ ]:
import numpy as np

# ── weights (sum to 1.0) ────────────────────────────────────────────────────
SALIENCE_ALPHA  = 0.40   # upvotes   – highest signal
SALIENCE_BETA   = 0.30   # likes
SALIENCE_GAMMA  = 0.20   # author_reputation
SALIENCE_DELTA  = 0.10   # log1p(views)  – dampened to avoid outlier dominance
SALIENCE_FLOOR  = 0.85   # min embedding scale (zero-engagement prompts keep 85 %)


def _norm(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    return (arr - mn) / (mx - mn + 1e-9)


def compute_salience(records: list, ids_order: list) -> np.ndarray:
    """
    Build a salience vector aligned to ids_order (the sorted id list
    produced by nomic_corpus_embs.py — same order as embs rows).

    Falls back to 0 for any missing field so the function is robust
    to older db.json snapshots.
    """
    meta_by_id = {r['id']: r for r in records}

    def _field(name, default=0):
        return np.array(
            [float(meta_by_id.get(pk, {}).get(name, default)) for pk in ids_order],
            dtype=np.float64
        )

    upvotes    = _norm(_field('upvotes'))
    likes      = _norm(_field('likes'))
    reputation = _norm(_field('author_reputation'))
    views      = _norm(np.log1p(_field('views')))

    raw = (
        SALIENCE_ALPHA * upvotes
        + SALIENCE_BETA  * likes
        + SALIENCE_GAMMA * reputation
        + SALIENCE_DELTA * views
    )
    return _norm(raw)   # re-normalise composite to [0, 1]


def apply_salience(embs: np.ndarray, salience: np.ndarray,
                   floor: float = SALIENCE_FLOOR) -> np.ndarray:
    """
    Scale each row by a factor in [floor, floor+1].
    High-salience docs expand their radius in embedding space → more
    kNN edges → higher Rayleigh-quotient λ after ArrowSpaceBuilder.build().
    """
    scale = floor + (1.0 - floor) * salience   # shape (N,)
    return embs * scale[:, np.newaxis]          # broadcast over dims


# ── load TinyDB records (needed for salience; dataset already loaded above) ─
db_records = db.all()   # list[dict] from db.json
salience   = compute_salience(db_records, ids)

print(f"Salience computed for {len(salience):,} prompts")
print(f"  min={salience.min():.4f}  max={salience.max():.4f}  "
      f"mean={salience.mean():.4f}  std={salience.std():.4f}")
print(f"  top-5 salience IDs:")
top5 = np.argsort(salience)[-5:][::-1]
for r in top5:
    print(f"    {ids[r]}  salience={salience[r]:.4f}")


In [ ]:
# ── build BOTH indexes for A/B comparison ──────────────────────────────────
from arrowspace import ArrowSpaceBuilder

GRAPH_PARAMS = {"eps": 1.0271119886118383, "k": 20, "topk": 10, "p": 2.0, "sigma": None}
BASE_SCALE   = 1.15   # keeps original tuned scaling

rows_base = embs * BASE_SCALE
rows_meta = apply_salience(embs * BASE_SCALE, salience, floor=SALIENCE_FLOOR)

print(f"Building aspace_base  ({len(rows_base):,} × {rows_base.shape[1]}d)...")
aspace_base, gl_base = ArrowSpaceBuilder().build(GRAPH_PARAMS, rows_base)
lambdas_base = aspace_base.lambdas()
print(f"  ✓ λ  min={min(lambdas_base):.6f}  max={max(lambdas_base):.6f}  "
      f"mean={sum(lambdas_base)/len(lambdas_base):.6f}")

print(f"\nBuilding aspace_meta  ({len(rows_meta):,} × {rows_meta.shape[1]}d)...")
aspace_meta, gl_meta = ArrowSpaceBuilder().build(GRAPH_PARAMS, rows_meta)
lambdas_meta = aspace_meta.lambdas()
print(f"  ✓ λ  min={min(lambdas_meta):.6f}  max={max(lambdas_meta):.6f}  "
      f"mean={sum(lambdas_meta)/len(lambdas_meta):.6f}")

print("\nBoth ArrowSpace indexes ready.")
print("  aspace_base  — pure semantic geometry (baseline)")
print("  aspace_meta  — semantic + engagement topology (metadata-weighted)")


# 🌐 Graph Laplacian Manifold — What You're Looking At

This surface is a **topographic map of your semantic space**.

Each point on the XY plane is a **centroid** (a cluster of embeddings from your dataset),
projected onto its first two principal components. The **height Z** is the diagonal of the
Graph Laplacian — the *degree* of that centroid in the ε-proximity graph built by ArrowSpace.

---

## 🔴 Red peaks = Dense semantic regions
High-degree centroids are **hubs**: many neighbours, tightly connected.
These are the dominant themes in your dataset — concepts the model understands well
and where retrieval is reliable.

## 🔵 Blue valleys = Sparse / frontier regions
Low-degree centroids are **outliers or boundary concepts**: few neighbours, weakly connected.
These are the *tail items* — the zone where standard cosine similarity loses precision
because there is not enough local structure to anchor results.

---

## Why this matters for search

A classical vector database treats the entire space as geometrically flat.
Every embedding is compared to a query with the same cosine formula, regardless
of whether it lives in a dense hub or an isolated valley.

**ArrowSpace does not.**

It builds this Laplacian, computes a per-item spectral score λ (the Rayleigh quotient),
and blends it with cosine similarity at query time:
score(q, item) = α · cosine(q, item) + (1-α) · λ_spectral(item)

text

Items in the valleys are re-weighted upward when they are genuinely relevant —
recovering the tail retrieval that flat cosine misses.

---

## What the shape of this surface tells you about your dataset

| Feature | Interpretation |
|---|---|
| Many tall red peaks | Strong thematic clusters — good dataset coverage |
| Very deep isolated blue holes | Potential coverage gaps or noisy embeddings |
| Smooth, gently rolling surface | Well-distributed, semantically coherent dataset |
| Sharp spikes (before clipping) | Outlier centroids — candidates for data cleaning |
| Surface changes over time | **Semantic drift** — your data distribution is shifting |

---

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# GRAPH LAPLACIAN — DATASET FINGERPRINT ANALYSIS
# ════════════════════════════════════════════════════════════════════════════
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# ── 1. Basic structural metrics ───────────────────────────────────────────
n          = L.shape[0]
nnz        = L.nnz
n_edges    = (nnz - n) // 2          # off-diagonal non-zeros / 2
sparsity   = 1.0 - nnz / (n * n)
avg_degree = float(degrees.mean())
std_degree = float(degrees.std())
max_degree = float(degrees.max())
min_degree = float(degrees.min())
degree_cv  = std_degree / avg_degree  # coefficient of variation: 0=flat, >1=spiky

# ── 2. Fiedler value (algebraic connectivity) ─────────────────────────────
# λ₁=0 always (constant vector); λ₂=Fiedler: how well-connected the graph is
try:
    diag_d     = np.array(L.diagonal(), dtype=np.float64)
    safe_d     = np.where(diag_d > 1e-12, diag_d, 1e-12)
    d_inv_sqrt = sp.diags(1.0 / np.sqrt(safe_d))
    L_norm     = d_inv_sqrt @ L @ d_inv_sqrt
    eigs       = spla.eigsh(L_norm, k=6, which="SM",
                            return_eigenvectors=False, tol=1e-5, maxiter=3000)
    eigs_sorted   = sorted(np.real(eigs))
    fiedler       = max(0.0, eigs_sorted[1])
    spectral_gap  = eigs_sorted[2] - eigs_sorted[1] if len(eigs_sorted) > 2 else 0.0
except Exception as e:
    fiedler, spectral_gap = 0.0, 0.0
    print(f"[warn] Fiedler computation failed: {e}")

# ── 3. Degree distribution buckets ────────────────────────────────────────
p10, p25, p50, p75, p90 = np.percentile(degrees, [10, 25, 50, 75, 90])
hub_count      = int((degrees > p90).sum())            # top 10% = hubs
isolated_count = int((degrees < p10).sum())            # bottom 10% = tails
hub_fraction   = hub_count / n * 100
tail_fraction  = isolated_count / n * 100

# ── 4. Connectivity interpretation ────────────────────────────────────────
def interpret_fiedler(f):
    if f < 0.01:  return "⚠️  Near-disconnected (multiple isolated clusters)"
    if f < 0.05:  return "🟡 Weakly connected (sparse bridge structure)"
    if f < 0.15:  return "🟢 Moderately connected (balanced topology)"
    return               "🔵 Strongly connected (dense, cohesive manifold)"

def interpret_cv(cv):
    if cv < 0.3:  return "Flat (uniform density, no dominant hubs)"
    if cv < 0.7:  return "Moderate variation (some hubs, mostly uniform)"
    if cv < 1.2:  return "High variation (clear hub/tail structure)"
    return               "Power-law-like (few mega-hubs, many isolated tails)"

def interpret_gap(gap):
    if gap < 0.02: return "Continuous spectrum (smooth manifold)"
    if gap < 0.1:  return "Soft cluster boundary (overlapping topics)"
    return                "Hard cluster boundary (distinct topic groups)"

# ── 5. Prepare Table Data ─────────────────────────────────────────────────
rows_table = [
    ("Nodes (centroids)",        f"{n}",                              "Clusters extracted from your embeddings"),
    ("Edges (connections)",      f"{n_edges:,}",                      "Semantic proximity links between centroids"),
    ("Sparsity",                 f"{sparsity*100:.2f}%",              "Most centroids are NOT directly connected"),
    ("Avg degree",               f"{avg_degree:.3f}",                 "Mean connectivity per centroid"),
    ("Degree std",               f"{std_degree:.3f}",                 "Spread of connectivity across centroids"),
    ("Degree CV",                f"{degree_cv:.3f}",                  interpret_cv(degree_cv)),
    ("Max degree (hub)",         f"{max_degree:.3f} (node {int(np.argmax(degrees))})", "Most connected centroid"),
    ("Min degree (tail)",        f"{min_degree:.3f} (node {int(np.argmin(degrees))})", "Most isolated centroid"),
    ("Hub centroids (top 10%)",  f"{hub_count} ({hub_fraction:.1f}% of nodes)",       "Dominant theme anchors"),
    ("Tail centroids (bot 10%)", f"{isolated_count} ({tail_fraction:.1f}% of nodes)", "Sparse / hard-to-retrieve zones"),
    ("Fiedler value λ₂",         f"{fiedler:.5f}",                    interpret_fiedler(fiedler)),
    ("Spectral gap λ₃−λ₂",       f"{spectral_gap:.5f}",               interpret_gap(spectral_gap)),
    ("eps parameter",            f"{GRAPH_PARAMS['eps']:.6f}",        "Proximity radius used to wire the graph"),
    ("k (neighbours)",           f"{GRAPH_PARAMS['k']}",              "Max neighbours per centroid"),
]

# ── 6. Print Plain Text Table ─────────────────────────────────────────────
print("\n" + "═" * 105)
print(" 📊 GRAPH LAPLACIAN — DATASET FINGERPRINT")
print("═" * 105)
print(f" {'METRIC':<25} │ {'VALUE':<20} │ {'INTERPRETATION'}")
print("─" * 105)
for row in rows_table:
    print(f" {row[0]:<25} │ {row[1]:<20} │ {row[2]}")
print("═" * 105)

# ── 7. One-paragraph narrative ────────────────────────────────────────────
print("\n── NARRATIVE SUMMARY ──────────────────────────────────────────────────────────")
print(f"""
Dataset contains {n} semantic centroids wired into a graph with {n_edges:,} edges 
({sparsity*100:.1f}% sparse).

Degree distribution: avg={avg_degree:.2f} ± {std_degree:.2f} (CV={degree_cv:.2f})
→ {interpret_cv(degree_cv)}

{hub_count} hub centroids ({hub_fraction:.1f}%) anchor the dominant themes.
{isolated_count} tail centroids ({tail_fraction:.1f}%) are the hard-to-retrieve zone —
exactly where ArrowSpace's spectral re-weighting adds value over flat cosine.

Algebraic connectivity (Fiedler λ₂ = {fiedler:.5f}):
→ {interpret_fiedler(fiedler)}

Spectral gap (λ₃ − λ₂ = {spectral_gap:.5f}):
→ {interpret_gap(spectral_gap)}
""")

# Graph Laplacian Fingerprint — What the Numbers Say

---

## The Dataset at a Glance

| | |
|---|---|
| **768 centroids** | semantic clusters automatically extracted from your embeddings |
| **4,791 edges** | proximity links — the wiring of your semantic space |
| **98.25% sparse** | the graph is lean: each centroid connects to ~7 neighbours on average |

---

## The Core Finding: Your Data Has a Power-Law Structure

The degree distribution has a **Coefficient of Variation of 4.13** — well into power-law territory.

This means your semantic space looks like this:
Degree
│ 266 ┤ █ ← node 69: mega-hub (one dominant concept cluster)

│ █

│ █ █

│ █ █ █

│ █ █ █ █ █

0 ┤─────────────────────────────────────► Centroids
few hubs long tail of isolated concepts

text

**A handful of concepts absorb most of the connections.  
Hundreds of concepts sit in near-isolation.**

This is not a data quality problem. It is the natural geometry of language —
the same power-law Zipf observed in word frequencies in 1935.

---

## Why This Is Exactly the Problem ArrowSpace Solves

Standard cosine similarity is **blind to this structure**.
It assigns the same retrieval logic to node 69 (degree 266) and node 51 (degree 0).

The result is predictable:

| Zone | What cosine does | What ArrowSpace does |
|---|---|---|
| 🔴 **Hub centroids** (top 10%, degree ≫ avg) | Works well — many neighbours reinforce the result | Works well + detects over-representation bias |
| 🔵 **Tail centroids** (bottom 10%, degree ≈ 0) | Retrieval fails silently — no neighbours to anchor results | Spectral re-weighting surfaces these items when genuinely relevant |


> **The tail is not noise. It is your users' hardest, most specific queries —
> the ones that matter most and fail most often.**

---

## The Graph Is Globally Healthy

**Fiedler value λ₂ = 0.649**

The Fiedler value measures algebraic connectivity — how hard it is to
"cut" the graph into disconnected pieces.

- Typical real-world graphs: **0.01 – 0.10**
- Your dataset: **0.649** → exceptionally well-connected

This tells you two things:

1. The dataset is **one coherent semantic territory**, not a fragmented archipelago
2. The ArrowSpace graph parameters (`eps=1.027, k=20`) are **well-calibrated** —
   the graph is neither over-wired nor broken

---

## The Manifold Is Continuous, Not Clustered

**Spectral gap λ₃ − λ₂ = 0.016**

A large spectral gap would indicate hard, well-separated topic clusters.
A near-zero gap (like yours) means the topics **blend gradually into each other** —
a smooth semantic manifold, not discrete buckets.

This has a direct consequence for retrieval:

> Hard clustering (k-means, topic models) **destroys information** on this dataset.  
> You need a method that respects the continuous geometry.  
> That is what ArrowSpace's taumode scoring does.

---

## The One Node Worth Inspecting

**Node 51 — degree = 0.000**

This centroid has zero connections in the proximity graph.
It represents a prompt (or group of prompts) that is **semantically unique** —
nothing in the dataset is similar enough to reach it under the current `eps`.

With flat cosine similarity: this item is **effectively invisible** to the retrieval system.  
With ArrowSpace: it can be **flagged, boosted, or used as a coverage gap signal**.

---

# 📈 Spectral λ Distribution — Reading the Shape of Your Data

---

## What the Plots Reveal

The two charts together tell a coherent story that links directly to the Laplacian fingerprint.

### Histogram — a bimodal-like shape with a flat plateau

The histogram shows a **sharp spike near λ ≈ 0** followed by a remarkably flat
plateau from λ ≈ 0.005 to λ ≈ 0.21, then a gradual red tail extending to λ = 1.0.

This is **not** a bell curve. It has three distinct regimes:
Count
│ ██ ← spike: 5,000 smooth items crammed near λ = 0

│ (hub-adjacent — cosine works fine here)

████████████████████████████ ← flat plateau: 10,000 mid items

│ (uniform structural density)

│ █████████████████████████████████ ← red tail: 5,000 rough items

│ (spectrally isolated — cosine misses these)
└──────────────────────────────────────────────► λ
0 0.05 0.21 1.0

text

The flat plateau is a key insight: mid-range items are **uniformly distributed**
across the spectral axis. This means there is no natural hard boundary between
"easy" and "hard" retrieval — the difficulty is a continuum.

---

### ECDF — the steepness tells the whole story

The ECDF rises steeply from 0 → 80% in the narrow range λ ∈ [0, 0.21],
then flattens into a long shallow tail from λ ∈ [0.21, 1.0].

| ECDF Region | λ Range | % Items | Interpretation |
|---|---|---|---|
| Steep rise | 0 → 0.21 | ~75% | Dense, hub-adjacent — fast to retrieve |
| Flat tail | 0.21 → 1.0 | ~25% | Sparse, isolated — where retrieval fails |

The **"elbow" at λ ≈ 0.21** is the natural operating boundary of cosine similarity.
Everything to the right of that elbow is the **ArrowSpace opportunity zone**.

---

## Linking Back to the Laplacian Fingerprint

| Laplacian Metric | Value | λ Distribution Echo |
|---|---|---|
| Degree CV | 4.131 (power-law) | λ CV = 1.311 — smoother, but still skewed |
| Fiedler λ₂ = 0.649 | Strong global cohesion | No gap in ECDF → no hard cluster boundary |
| Spectral gap = 0.016 | Continuous manifold | Flat histogram plateau confirms smooth topology |
| Hub centroids (10%) | 77 mega-hubs | The λ ≈ 0 spike — 5,000 items anchored to hubs |
| Tail centroids (10%) | 77 isolated nodes | The red ECDF tail — 5,000 items with λ > 0.21 |

> **Key insight:** The degree CV (4.131) is 3× higher than the λ CV (1.311).
> This compression is spectral smoothing in action — the Laplacian integrates
> neighbourhood structure and dampens the raw degree extremes into a more
> continuous signal. ArrowSpace scores operate on this smoothed geometry,
> not on the raw noisy degree counts.

---

## The Retrieval Gap — Quantified
Total items : 20,000
────────────────────────────────────────────────────
Cosine covers well: 15,000 items (smooth + mid)
Cosine misses: 5,000 items (rough zone, λ > 0.215)
──────
25% of your dataset is invisible to standard search

text

Those 5,000 rough items are not low-quality data.
They are your **most specific, most unique, hardest-to-match prompts** —
exactly the ones power users query, and exactly the ones that return
poor results today.

ArrowSpace's taumode blends the Rayleigh quotient λ into the similarity score
at query time, pulling rough items up in the ranking when they are
genuinely the best match — without penalising smooth items.

---

In [ ]:
dataset[0]

In [ ]:

q_embs = np.load(query_path).astype(np.float64)
idx    = json.load(open(query_index_path))

row    = idx["q_09"]["row_index"]          # 8
query  = q_embs[row].astype(np.float64)

In [ ]:
ids_path = ROOT / "data" / "nomic_embs" / "embeddings_nomic_structured_768d_ids.npy"

## A/B Evaluation — Baseline vs. Metadata-Weighted

Run both indexes on the evaluation queries and compare **MRR** and **Precision@1**.
This satisfies the challenge's explicit A/B comparison requirement.


In [ ]:
# ── A/B evaluation helper ───────────────────────────────────────────────────
import numpy as np, json
from IPython.display import display, HTML

q_embs_eval = np.load(query_path).astype(np.float64)
idx_eval    = json.load(open(query_index_path))

ALPHA_EVAL = 0.70
TOPK_EVAL  = 10

def run_eval(aspace_obj, gl_obj, label: str):
    rr_list, p1_list = [], []
    for qid, meta in idx_eval.items():
        q_vec  = q_embs_eval[meta['row_index']].astype(np.float64)
        target = meta['expected_prompt_id']
        hits   = aspace_obj.search(q_vec, gl_obj, ALPHA_EVAL)[:TOPK_EVAL]
        rank   = next((r for r, (ri, _) in enumerate(hits, 1) if ids[ri] == target), None)
        rr_list.append(1.0 / rank if rank else 0.0)
        p1_list.append(1.0 if rank == 1 else 0.0)
    return {
        'label'   : label,
        'MRR'     : float(np.mean(rr_list)),
        'P@1'     : float(np.mean(p1_list)),
        'rr_list' : rr_list,
    }

res_base = run_eval(aspace_base, gl_base, 'aspace_base (semantic only)')
res_meta = run_eval(aspace_meta, gl_meta, 'aspace_meta (+ salience)')

delta_mrr = res_meta['MRR'] - res_base['MRR']
delta_p1  = res_meta['P@1'] - res_base['P@1']

BG2='#0f1117'; SURF3='#1a1d27'; TXT2='#e2e6f0'; MUT2='#7a82a0'
GRN2='#6daa45'; RED2='#dd6974'; YLW2='#e8b84b'; TEAL2='#4f98a3'

def _delta_badge(d):
    col  = GRN2 if d > 0 else (RED2 if d < 0 else MUT2)
    sign = '+' if d >= 0 else ''
    return f'<span style="color:{col};font-weight:700">{sign}{d:.4f}</span>'

rows_html = ''
for res in [res_base, res_meta]:
    rows_html += f"""
<tr style="border-bottom:1px solid #2e3350">
  <td style="padding:10px 14px;color:{TXT2};font-family:monospace">{res['label']}</td>
  <td style="padding:10px 14px;text-align:center;font-family:monospace;color:{TEAL2};font-weight:700">{res['MRR']:.4f}</td>
  <td style="padding:10px 14px;text-align:center;font-family:monospace;color:{TEAL2};font-weight:700">{res['P@1']:.4f}</td>
</tr>"""

rows_html += f"""
<tr style="background:#1a2e1a">
  <td style="padding:10px 14px;color:{MUT2};font-style:italic">Δ (meta − base)</td>
  <td style="padding:10px 14px;text-align:center">{_delta_badge(delta_mrr)}</td>
  <td style="padding:10px 14px;text-align:center">{_delta_badge(delta_p1)}</td>
</tr>"""

html_out = f"""
<div style="background:{BG2};padding:20px;border-radius:12px;font-family:system-ui,sans-serif">
  <h3 style="color:{TXT2};margin:0 0 12px">A/B Evaluation · top-{TOPK_EVAL} · α={ALPHA_EVAL}</h3>
  <table style="width:100%;border-collapse:collapse;background:{SURF3};border-radius:8px;overflow:hidden">
    <tr style="background:#22263a">
      <th style="padding:8px 14px;color:{MUT2};font-size:12px;text-align:left">Index</th>
      <th style="padding:8px 14px;color:{MUT2};font-size:12px;text-align:center">MRR</th>
      <th style="padding:8px 14px;color:{MUT2};font-size:12px;text-align:center">P@1</th>
    </tr>
    {rows_html}
  </table>
  <p style="color:{MUT2};font-size:12px;margin:10px 0 0">
    Δ = metadata-weighted minus baseline. Positive = salience improves retrieval.
  </p>
</div>"""
display(HTML(html_out))
